# 1. Preparación, entrenamiento y trazas
Ejecutar en Linux con el kernel de `.venv`. Los datos y modelos quedan locales e ignorados por Git. Este notebook llama a los adaptadores del protocolo; los notebooks históricos son solo referencia.


In [1]:
from pathlib import Path
import platform
import sys
import os
raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'shared' / 'contract.py').is_file()), None)
assert raiz is not None, f'No se encuentra la raíz del repositorio desde {Path.cwd()}'
os.chdir(raiz)
sys.path.insert(0, str(raiz))
from shared.contract import ROOT, validate_splits
assert Path.cwd().resolve() == ROOT
print(platform.platform(), sys.version.split()[0], ROOT)
assert sys.platform.startswith('linux'), 'Entrenar LSTM en el entorno Linux validado'
import tensorflow as tf
import lightgbm
import pyarrow
print('Dependencias:', 'tensorflow', tf.__version__, 'lightgbm', lightgbm.__version__, 'pyarrow', pyarrow.__version__)


Linux-5.15.148-tegra-aarch64-with-glibc2.35 3.10.12 /home/jetson/Desktop/greenhouse-online-anomaly-detection
Dependencias: tensorflow 2.16.1 lightgbm 4.6.0 pyarrow 25.0.1


## Datos físicos congelados
Si faltan los archivos locales, revisar primero su tamaño y procedencia en `data_contract/manifests/imported_files.json`; la hidratación verifica hashes y no modifica la fuente.


In [2]:
# Ejecutar solo si faltan los archivos ignorados y existe el checkout fuente local.
# from shared.data_preparation.hydrate import hydrate
# hydrate(Path('../Clasificaci-n-y-correcci-n'))
contract = validate_splits()
assert [contract['splits'][s]['rows'] for s in ('train', 'validation', 'test')] == [172873, 28800, 31680]
print(contract['data_source'], contract['manifest_id'])


Agroconnect 4cc10ef96d923a326a86c0882a356a4d09d35a6da6553bfdbb60dd80a5f0737c


## LightGBM con noche y contexto interior/exterior
Es un experimento nuevo de 68 variables. El v8 original permanece como referencia de 66 variables.


In [3]:
from approaches.lightgbm.night_context import train as train_lightgbm, export_test as export_lightgbm
lightgbm_run = train_lightgbm()
lightgbm_trace, lightgbm_isolated = export_lightgbm(lightgbm_run)
print(lightgbm_run, lightgbm_trace, lightgbm_isolated, sep='\n')


/home/jetson/Desktop/greenhouse-online-anomaly-detection/experiments/runs/lightgbm_night_v1/20261105T093300Z
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lightgbm_night_test.parquet
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lightgbm_night_model2_isolated_test.parquet


## LSTM causal
Model 1 y Model 2 se entrenan con train; validation selecciona checkpoints y umbral. Test solo se usa al exportar trazas. Las ventanas se generan por lotes para no copiar todo train a la GPU. Puede tardar bastante. Si LightGBM ya terminó y existen sus trazas locales, tras reiniciar el kernel se pueden ejecutar las celdas de entorno y datos y continuar directamente aquí.


In [4]:
from approaches.lstm.adapter import train as train_lstm, export_test as export_lstm
lstm_run = train_lstm()
lstm_trace, lstm_isolated = export_lstm(lstm_run)
print(lstm_run, lstm_trace, lstm_isolated, sep='\n')


Epoch 1/50


2026-11-05 10:35:29.630261: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:35:29.630392: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
2026-11-05 10:35:32.675576: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:35:32.675942: W tensorflow/core/framework/local_rendezvous.cc

676/676 - 33s - 49ms/step - accuracy: 0.7710 - loss: 0.5050 - pr_auc: 0.5718 - precision: 0.3486 - recall: 0.7116 - roc_auc: 0.8289 - val_accuracy: 0.9401 - val_loss: 0.2186 - val_pr_auc: 0.8797 - val_precision: 0.8635 - val_recall: 0.7545 - val_roc_auc: 0.9506 - learning_rate: 0.0010
Epoch 2/50


2026-11-05 10:35:50.710020: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:35:50.710169: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]


676/676 - 20s - 29ms/step - accuracy: 0.9412 - loss: 0.2010 - pr_auc: 0.9260 - precision: 0.7472 - recall: 0.8835 - roc_auc: 0.9722 - val_accuracy: 0.9701 - val_loss: 0.0977 - val_pr_auc: 0.9543 - val_precision: 0.9724 - val_recall: 0.8416 - val_roc_auc: 0.9816 - learning_rate: 0.0010
Epoch 3/50


2026-11-05 10:35:52.353405: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:35:52.353496: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/Shape/_6]]
2026-11-05 10:36:12.034385: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:36:12.034500: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_2]]


... (55 outputs left out)

2026-11-05 10:42:56.368329: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:42:56.368431: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_28]]
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
2026-11-05 10:42:56.699962: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:42:56.700054: W tensorflow/core/framework/local_rendezvous.c

96/96 - 3s - 33ms/step - accuracy: 0.9574 - loss: 0.1019 - val_accuracy: 0.9180 - val_loss: 0.4304 - val_macro_f1: 0.9202 - learning_rate: 6.2500e-05
Epoch 18/50


2026-11-05 10:43:01.607810: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-11-05 10:43:01.607905: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/Shape/_4]]
/usr/local/lib/python3.10/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
2026-11-05 10:43:01.927878: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[StatefulPartitionedCall/Shape/_4]]
2026-11-05 10:43:01.92

96/96 - 5s - 55ms/step - accuracy: 0.9580 - loss: 0.1018 - val_accuracy: 0.9182 - val_loss: 0.4265 - val_macro_f1: 0.9205 - learning_rate: 6.2500e-05
/home/jetson/Desktop/greenhouse-online-anomaly-detection/experiments/runs/lstm_causal_v2/20261105T093456Z
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lstm_test.parquet
/home/jetson/Desktop/greenhouse-online-anomaly-detection/results/traces/lstm_model2_isolated_test.parquet


Guardar las rutas de ambos runs. El notebook 02 verifica la paridad de objetivos y genera el informe comparativo. No versionar las salidas de este notebook.
